In [ ]:
import numpy as np
import pandas as pd

import sys
sys.path.append('../')

from src.data_utils import *
from src.tda_utils import *
from src.utils import *

import plotly.express as px
import plotly.graph_objects as go

import glob

In [ ]:
# list_data_name = ["BTCUSDT_1d", "ETHUSDT_1d", "BNBUSDT_1d", "XRPUSDT_1d", "ADAUSDT_1d", "LTCUSDT_1d"]                   
# list_data_name = ["BTCUSDT_1h", "ETHUSDT_1h", "BNBUSDT_1h", "XRPUSDT_1h", "ADAUSDT_1h", "LTCUSDT_1h"]                   

# list_data_name = ["ADAUSDT_1d"]                   
list_data_name = ["ADAUSDT_1h"]                   
                        
from_date = '2021-12-31 00:00:00' # 2020-12-31 00:00:00
until_date = '2025-07-31 00:00:00'

In [ ]:
initial_slice_position = 0
list_slice_size_in_days = [3, 7, 14, 21, 30, 45]
# list_slice_size_in_days = [45]
# list_slice_size_in_days = [14, 21, 30]
num_point_for_1d = (1 * 24) # 1h
column_name = 'processed_log_return_wtmra_0'
save = True

In [ ]:
# Parameters for the point cloud creation
list_dimension = [2, 3, 4]
list_time_delay = [3, 7]
# time_delay_in_days = 7
stride = 1

In [ ]:
# Create folder for saving results if doesn't exist
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

def erase_files(path): 
    files = glob.glob(path + '*')
    for f in files:
        os.remove(f)

In [ ]:
# Validation function for Takens embedding parameters
def validate_and_suggest_embedding_params(slice_size, dimension, time_delay_in_days, data_name=""):
    """
    Validate if the embedding parameters are compatible with the slice size.
    If not, suggest alternative parameters that would work.
    
    Returns:
        dict: {'valid': bool, 'message': str, 'suggestions': dict}
    """
    min_required_points = (dimension - 1) * time_delay_in_days + 1
    
    if slice_size >= min_required_points:
        return {
            'valid': True, 
            'message': f"✅ Parameters are valid for {data_name}. Slice size: {slice_size}, Required: {min_required_points}",
            'suggestions': {}
        }
    
    # Calculate alternative parameters
    suggestions = {}
    
    # Option 1: Reduce time delay to fit current dimension
    max_time_delay = (slice_size - 1) // (dimension - 1)
    if max_time_delay > 0:
        suggestions['reduce_time_delay'] = {
            'dimension': dimension,
            'time_delay_in_days': max_time_delay,
            'required_points': (dimension - 1) * max_time_delay + 1
        }
    
    # Option 2: Reduce dimension to fit current time delay
    max_dimension = ((slice_size - 1) // time_delay_in_days) + 1
    if max_dimension >= 2:
        suggestions['reduce_dimension'] = {
            'dimension': max_dimension,
            'time_delay_in_days': time_delay_in_days,
            'required_points': (max_dimension - 1) * time_delay_in_days + 1
        }
    
    # Option 3: Balanced approach - moderate dimension and time delay
    if slice_size > 100:  # Only suggest if we have reasonable amount of data
        balanced_dim = 2
        balanced_delay = min(time_delay_in_days, (slice_size - 1) // (balanced_dim - 1))
        if balanced_delay > 0:
            suggestions['balanced'] = {
                'dimension': balanced_dim,
                'time_delay_in_days': balanced_delay,
                'required_points': (balanced_dim - 1) * balanced_delay + 1
            }
    
    message = f"❌ Parameters incompatible for {data_name}!\n"
    message += f"   Slice size: {slice_size}, Required: {min_required_points}\n"
    message += f"   Current: dimension={dimension}, time_delay={time_delay_in_days} days\n"
    
    if suggestions:
        message += "   💡 Suggested alternatives:"
        for key, params in suggestions.items():
            message += f"\n   - {key.replace('_', ' ').title()}: dim={params['dimension']}, delay={params['time_delay_in_days']} (needs {params['required_points']} points)"
    
    return {
        'valid': False,
        'message': message,
        'suggestions': suggestions
    }

def auto_fix_embedding_params(slice_size, dimension, time_delay_in_days, strategy='balanced'):
    """
    Automatically fix embedding parameters using the specified strategy.
    
    Args:
        slice_size: Number of data points in the slice
        dimension: Current embedding dimension
        time_delay_in_days: Current time delay
        strategy: 'reduce_time_delay', 'reduce_dimension', or 'balanced'
    
    Returns:
        tuple: (new_dimension, new_time_delay_in_days)
    """
    validation = validate_and_suggest_embedding_params(slice_size, dimension, time_delay_in_days)
    
    if validation['valid']:
        return dimension, time_delay_in_days
    
    suggestions = validation['suggestions']
    
    if strategy in suggestions:
        params = suggestions[strategy]
        return params['dimension'], params['time_delay_in_days']
    elif 'balanced' in suggestions:
        params = suggestions['balanced']
        return params['dimension'], params['time_delay_in_days']
    elif 'reduce_dimension' in suggestions:
        params = suggestions['reduce_dimension']
        return params['dimension'], params['time_delay_in_days']
    elif 'reduce_time_delay' in suggestions:
        params = suggestions['reduce_time_delay']
        return params['dimension'], params['time_delay_in_days']
    else:
        # Fallback to minimal working parameters
        return 2, max(1, (slice_size - 1) // 1)

In [ ]:
for data_name in list_data_name:
    for slice_size_in_days in list_slice_size_in_days:
        slice_size = slice_size_in_days * num_point_for_1d
        for time_delay in list_time_delay:
            time_delay_in_days = (time_delay * 24)
            for dimension in list_dimension:

                # ==================== PARAMETER VALIDATION ====================
                print(f"\n🔍 Validating parameters for {data_name}:")
                print(f"   Slice size: {slice_size} points ({slice_size_in_days} days)")
                print(f"   Original parameters: dimension={dimension}, time_delay={time_delay_in_days} days")
                
                # Validate embedding parameters
                validation_result = validate_and_suggest_embedding_params(
                    slice_size, dimension, time_delay_in_days, data_name
                )
                
                # Use original or auto-fixed parameters
                if validation_result['valid']:
                    final_dimension = dimension
                    final_time_delay_in_days = time_delay_in_days
                    print(validation_result['message'])
                else:
                    print(validation_result['message'])
                    
                    # Auto-fix parameters using reduce_time_delay strategy to keep original dimension
                    final_dimension, final_time_delay_in_days = auto_fix_embedding_params(
                        slice_size, dimension, time_delay_in_days, strategy='reduce_time_delay'
                    )
                    
                    print(f"   🔧 Auto-fixed parameters: dimension={final_dimension}, time_delay={final_time_delay_in_days} days")
                    print(f"   ✅ New requirement: {(final_dimension - 1) * final_time_delay_in_days + 1} points (vs {slice_size} available)")
                
                # ============================================================

                # Load the data
                data = pd.read_csv(f'../data/01-output-{data_name}-from-{from_date}-until-{until_date}-log-return.csv', parse_dates=['date'], index_col='date')

                df = data[column_name]

                # Update paths to include final parameters
                full_series_with_highlight_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/full_series_with_highlight/'
                sliced_time_series_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/sliced_time_series/'
                point_cloud_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/point_cloud/'
                persistence_diagram_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/persistence_diagram/'
                mapper_graph_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/mapper_graph/'
                mosaic_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/mosaic/'

                features_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/features/'
                features_plot_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/features_plot/'
                mosaic_and_features_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/mosaic_and_features/'
                video_path = f'../results/{column_name}_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}/video/'


                if save:
                    create_folder(full_series_with_highlight_path)
                    create_folder(sliced_time_series_path)
                    create_folder(point_cloud_path)
                    create_folder(persistence_diagram_path)
                    create_folder(mapper_graph_path)
                    create_folder(mosaic_path)
                    create_folder(features_path)
                    create_folder(features_plot_path)
                    create_folder(mosaic_and_features_path)
                    create_folder(video_path)

                if save:
                    erase_files(full_series_with_highlight_path)
                    erase_files(sliced_time_series_path)
                    erase_files(point_cloud_path)
                    erase_files(persistence_diagram_path)
                    erase_files(mapper_graph_path)
                    erase_files(mosaic_path)
                    erase_files(features_path)
                    erase_files(features_plot_path)
                    erase_files(mosaic_and_features_path)
                    erase_files(video_path)

                df_features = pd.DataFrame()

                # Create an empty dataframe with these columns: [initial_slice_position, slice_size, start_date, end_date, connected_components_entropy, loops_entropy, voids_entropy, connected_components_amplitude, loops_amplitude, voids_amplitude, connected_components_number_of_points, loops_number_of_points, voids_number_of_points]
                df_features = pd.DataFrame(columns=['initial_slice_position', 'slice_size', 'start_date', 'end_date', 'connected_components_entropy', 'loops_entropy', 'voids_entropy', 'connected_components_amplitude', 'loops_amplitude', 'voids_amplitude', 'connected_components_number_of_points', 'loops_number_of_points', 'voids_number_of_points'])

                for i in range(0, (len(data)-slice_size), num_point_for_1d):

                    # fig = generate_plot_full_series_with_highlight(df, initial_slice_position + i, slice_size, save, full_series_with_highlight_path, f'full_series_with_highlight_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}', data_name)
                    time_series = get_sliced_time_series(df, initial_slice_position + i, slice_size)
                    # fig = generate_plot_sliced_time_series(df, initial_slice_position + i, slice_size, save, sliced_time_series_path, f'sliced_time_series_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
                    # fig.show()
                    point_cloud = create_point_cloud(time_series, final_dimension, final_time_delay_in_days, stride)
                    # fig = generate_plot_point_cloud(point_cloud, initial_slice_position + i, slice_size, save, point_cloud_path, f'point_cloud_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
                    # fig.show()
                    # fig = generate_plot_persistence_diagram(point_cloud, initial_slice_position + i, slice_size, save, persistence_diagram_path, f'persistence_diagram_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
                    # fig.show()
                    # fig = plot_persistence_barcode(point_cloud, initial_slice_position + i, slice_size, save, '../results/persistence_barcode/', f'persistence_barcode_{initial_slice_position + i}_{slice_size}')
                    # fig.show()
                    # fig = plot_persistence_heatmap(point_cloud, initial_slice_position + i, slice_size, save, '../results/persistence_heatmap/', f'persistence_heatmap_{initial_slice_position + i}_{slice_size}')
                    # fig.show()
                    # fig = plot_persistence_landscape(point_cloud, initial_slice_position + i, slice_size, save, '../results/persistence_landscape/', f'persistence_landscape_{initial_slice_position + i}_{slice_size}')
                    # fig.show()
                    # fig = generate_plot_mapper_graph(point_cloud, initial_slice_position + i, slice_size, save, mapper_graph_path, f'mapper_graph_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
                    # fig.show()

                    entropy_features_dict, amplitude_features_dict, number_of_points_features_dict = get_features(point_cloud)

                    # Add the features to the dataframe using concat
                    df_features = pd.concat([df_features, pd.DataFrame({
                        'initial_slice_position': initial_slice_position + i,
                        'slice_size': slice_size,
                        'start_date': data.index[initial_slice_position + i],
                        'end_date': data.index[initial_slice_position + i + slice_size],
                        'connected_components_entropy': entropy_features_dict['connected_components_entropy'],
                        'loops_entropy': entropy_features_dict['loops_entropy'],
                        'voids_entropy': entropy_features_dict['voids_entropy'],
                        'connected_components_amplitude': amplitude_features_dict['connected_components_amplitude'],
                        'loops_amplitude': amplitude_features_dict['loops_amplitude'],
                        'voids_amplitude': amplitude_features_dict['voids_amplitude'],
                        'connected_components_number_of_points': number_of_points_features_dict['connected_components_number_of_points'],
                        'loops_number_of_points': number_of_points_features_dict['loops_number_of_points'],
                        'voids_number_of_points': number_of_points_features_dict['voids_number_of_points']
                    }, index=[0])], ignore_index=True)        

                    # full_path_img1 = f'{sliced_time_series_path}/sliced_time_series_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                    # full_path_img2 = f'{full_series_with_highlight_path}/full_series_with_highlight_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                    # full_path_img3 = f'{point_cloud_path}/point_cloud_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                    # full_path_img4 = f'{point_cloud_path}/point_cloud_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}_fixed_axis.png'
                    # full_path_img5 = f'{persistence_diagram_path}/persistence_diagram_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                    # full_path_img6 = f'{persistence_diagram_path}/persistence_diagram_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}_fixed_axis.png'
                    # full_path_img7 = f'{mapper_graph_path}/mapper_graph_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                    # full_path_mosaic = f'{mosaic_path}/mosaic_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                    
                    # generate_mosaic(full_path_img1, full_path_img2, full_path_img3, full_path_img4, full_path_img5, full_path_img6, full_path_img7, full_path_mosaic)                

                # Save the dataframe with the features
                if save:
                    df_features.to_csv(f'{features_path}/features_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}.csv', index=False)
                    print(f'Features saved to {features_path}/features_{data_name}_time_delay_{final_time_delay_in_days}_slice_size_{slice_size}_dimension_{final_dimension}.csv')

                # df_features = pd.read_csv(f'{features_path}/features_{data_name}_time_delay_{time_delay_in_days}_slice_size_{slice_size}.csv')

                # list_columns_to_normalize = ['connected_components_entropy', 'loops_entropy', 'voids_entropy',
                #     'connected_components_amplitude', 'loops_amplitude', 'voids_amplitude',
                #     'connected_components_number_of_points', 'loops_number_of_points',
                #     'voids_number_of_points'
                #     ]
                # df_features_normalized = normalize_dataframe(df_features, list_columns_to_normalize)
                # df_features_normalized_v2 = normalize_dataframe(df_features, list_columns_to_normalize, (-1, 1))

                # for i in range(0, (len(data)-slice_size), num_point_for_1d):
                #     mosaic_path_img = f'{mosaic_path}/mosaic_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                #     features_path_img = f'{features_plot_path}/features_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                #     features_accumulate_path_img = f'{features_plot_path}/features_accumulate_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                #     mosaic_and_features_path_img = f'{mosaic_and_features_path}/mosaic_and_features_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}.png'
                #     fig = generate_features_plot(data, column_name, df_features_normalized_v2, initial_slice_position + i, slice_size, save, features_plot_path, f'features_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
                #     # fig = generate_features_plot(data, column_name, df_features_normalized_accumulated, initial_slice_position + i, slice_size, save, features_plot_path, f'features_accumulate_{generate_numerical_filename(initial_slice_position + i, len(data))}_{slice_size}')
                #     generate_mosaic_and_features(mosaic_path_img, features_path_img, mosaic_and_features_path_img)

                # create_mosaic_video(f'{mosaic_and_features_path}/', f'{video_path}/', f'mosaic_and_features_video_{slice_size}.mp4')